# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant pandas matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata and print summary
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and all `@id`s from the dataset.

In [ ]:
# Inspect available record sets and their fields by @id
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for i, rs in enumerate(record_sets):
    print(f"Record Set {i+1}:")
    print(f"  @id           : {rs.id}")
    print(f"  name          : {rs.name}")
    print(f"  description   : {rs.description}")
    # List all fields with their ids
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Record set and field `@id`s are used for all references.

In [ ]:
# Extract all record set @ids
record_set_ids = [rs.id for rs in record_sets]
print("Record Set IDs:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")
    print("Fields:", df.columns.tolist())
    print()
# For demonstration, choose the main survey record set
# We assume the main demographic/survey record set is the first one (adjust if needed)
main_record_set_id = record_set_ids[0]
print(f"Using main record set: {main_record_set_id}")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Here we perform basic exploration, filtering, normalization, and grouping on the `phq9_total` field (PHQ-9 score), using only `@id` references inline.

We use the following example fields (replace with those listed in Section 2 if record set field `@id`s differ):
- `phq9_total` (e.g., `_:phq9_total`), assumed integer total PHQ-9 score for depression.
- `gad7_total` (generalized anxiety disorder score), `age`, or similar fields.
- `gender` (`@id` may be `_:gender`), used for grouping.

In [ ]:
# You can edit these field names to match their actual @id values if different.
numeric_field_id = None
group_field_id = None
# Intelligent scan for typical total score fields and group candidates
columns = dataframes[main_record_set_id].columns
print("Available columns:\n", columns)
for col in columns:
    if col.lower().startswith("phq9"): # e.g., 'phq9_total', '@id:phq9_total', etc
        numeric_field_id = col
    if "gender" in col.lower():
        group_field_id = col

if numeric_field_id is None:
    numeric_field_id = columns[0]  # fallback
print(f"Numeric field selected for analysis: {numeric_field_id}")
if group_field_id:
    print(f"Grouping field: {group_field_id}")

# Some PHQ9 scores may be stored as strings, convert to numeric safely
df = dataframes[main_record_set_id].copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = 10
# Filter for scores above threshold (e.g., moderate depression)
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):\n")
display(filtered_df.head())

# Normalize the numeric field for comparison
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group analysis (e.g., by gender)
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = (filtered_df.groupby(group_field_id)[numeric_field_id]
                 .agg(['mean','count','std']))
    print(f"Grouped data by {group_field_id} (mean/size/std):")
    display(grouped_df.head())

## 5. Visualization
Show the distribution of PHQ-9 total scores and a boxplot by gender (or available group field).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
- We successfully loaded the FAIR² Mental Health Survey dataset via its Croissant schema using `mlcroissant`.
- We identified available record sets and fields by their `@id`s and explored summary statistics and visualizations for the PHQ-9 (depression) score.
- The workflow presented here can be adapted for further feature engineering, predictive modeling, or broader demographic analysis using the dataset and its extensive schema.

For more information on the dataset, refer to the [FAIR² Mental Health Survey Dataset Croissant schema](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).